In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


class AcademicPerformanceAnalyzer:

    # ============================================================
    # Q1. CREATE ACADEMIC DATAFRAME
    # ============================================================

    def create_academic_df(self, academic_data: list) -> pd.DataFrame:

        columns = [
            "StudentID",
            "StudentName",
            "Department",
            "Semester",
            "Subject",
            "Marks",
            "Attendance",
            "ExamType"
        ]

        df = pd.DataFrame(academic_data, columns=columns)

        return df


    # ============================================================
    # Q2. ACADEMIC EDA
    # ============================================================

    def academic_eda(self, df: pd.DataFrame) -> dict:

        result = {
            "number_of_records": len(df),
            "number_of_students": df["StudentID"].nunique(),
            "number_of_departments": df["Department"].nunique(),
            "number_of_subjects": df["Subject"].nunique(),
            "number_of_final_exams": (df["ExamType"] == "Final").sum(),
            "number_of_midterms": (df["ExamType"] == "Midterm").sum()
        }

        return result


    # ============================================================
    # Q3. MISSING VALUE ANALYSIS
    # ============================================================

    def missing_academic_summary(self, df: pd.DataFrame) -> pd.DataFrame:

        result = df.isnull().sum().to_frame("MissingValues")

        return result


    # ============================================================
    # Q4. CLEAN ACADEMIC DATA
    # ============================================================

    def clean_academic_data(self, df: pd.DataFrame) -> pd.DataFrame:

        # Remove duplicate StudentID + Subject + ExamType
        df = df.drop_duplicates(
            subset=["StudentID", "Subject", "ExamType"]
        )

        # Remove Marks <= 0
        df = df[df["Marks"] > 0]

        # Remove Attendance outside 0–100
        df = df[
            (df["Attendance"] >= 0) &
            (df["Attendance"] <= 100)
        ]

        # Reset index
        df = df.reset_index(drop=True)

        return df


    # ============================================================
    # Q5. ADD GRADE BAND
    # ============================================================

    def add_grade_band(self, df: pd.DataFrame) -> pd.DataFrame:

        conditions = [
            df["Marks"] >= 90,
            df["Marks"] >= 80,
            df["Marks"] >= 70,
            df["Marks"] >= 60,
            df["Marks"] >= 50,
            df["Marks"] < 50
        ]

        choices = [
            "A+",
            "A",
            "B",
            "C",
            "D",
            "F"
        ]

        df["GradeBand"] = np.select(
            conditions,
            choices,
            default="F"
        )

        return df


    # ============================================================
    # Q6. ADD ATTENDANCE FLAG
    # ============================================================

    def add_attendance_flag(
        self,
        df: pd.DataFrame,
        threshold: float
    ) -> pd.DataFrame:

        df["AttendanceFlag"] = np.where(
            df["Attendance"] >= threshold,
            1,
            0
        )

        return df


    # ============================================================
    # Q7. STUDENT SUMMARY
    # ============================================================

    def student_summary(self, df: pd.DataFrame) -> pd.DataFrame:

        summary = (
            df.groupby("StudentID")
            .agg(
                AverageMarks=("Marks", "mean"),
                MaximumMarks=("Marks", "max"),
                MinimumMarks=("Marks", "min"),
                AverageAttendance=("Attendance", "mean"),
                SubjectCount=("Subject", "nunique")
            )
            .reset_index()
        )

        # Add student name
        names = (
            df[["StudentID", "StudentName"]]
            .drop_duplicates("StudentID")
        )

        summary = summary.merge(
            names,
            on="StudentID",
            how="left"
        )

        # Arrange columns
        summary = summary[
            [
                "StudentID",
                "StudentName",
                "AverageMarks",
                "MaximumMarks",
                "MinimumMarks",
                "AverageAttendance",
                "SubjectCount"
            ]
        ]

        # Sort descending
        summary = summary.sort_values(
            "AverageMarks",
            ascending=False
        ).reset_index(drop=True)

        return summary


    # ============================================================
    # Q8. SUBJECT SUMMARY
    # ============================================================

    def subject_summary(self, df: pd.DataFrame) -> pd.DataFrame:

        summary = (
            df.groupby("Subject")
            .agg(
                StudentCount=("StudentID", "nunique"),
                AverageMarks=("Marks", "mean"),
                HighestMarks=("Marks", "max"),
                LowestMarks=("Marks", "min"),
                AverageAttendance=("Attendance", "mean")
            )
            .reset_index()
        )

        # Sort average marks ascending
        summary = summary.sort_values(
            "AverageMarks",
            ascending=True
        ).reset_index(drop=True)

        return summary


    # ============================================================
    # Q9. DEPARTMENT SUMMARY
    # ============================================================

    def department_summary(self, df: pd.DataFrame) -> pd.DataFrame:

        summary = (
            df.groupby("Department")
            .agg(
                StudentCount=("StudentID", "nunique"),
                AverageMarks=("Marks", "mean"),
                AverageAttendance=("Attendance", "mean"),
                HighestMarks=("Marks", "max")
            )
            .reset_index()
        )

        # Sort by average marks descending
        summary = summary.sort_values(
            "AverageMarks",
            ascending=False
        ).reset_index(drop=True)

        return summary


    # ============================================================
    # Q10. NUMPY MARKS STATISTICS
    # ============================================================

    def numpy_marks_statistics(
        self,
        df: pd.DataFrame
    ) -> tuple:

        marks = df["Marks"].to_numpy()

        mean = np.mean(marks)
        median = np.median(marks)
        standard_deviation = np.std(marks)
        minimum = np.min(marks)
        maximum = np.max(marks)
        percentile_25 = np.percentile(marks, 25)
        percentile_75 = np.percentile(marks, 75)

        return (
            mean,
            median,
            standard_deviation,
            minimum,
            maximum,
            percentile_25,
            percentile_75
        )


    # ============================================================
    # Q11. ABOVE SUBJECT AVERAGE
    # ============================================================

    def above_subject_average(
        self,
        df: pd.DataFrame
    ) -> pd.DataFrame:

        # Calculate average marks for each subject
        df["SubjectAverage"] = (
            df.groupby("Subject")["Marks"]
            .transform("mean")
        )

        # Keep only marks above subject average
        result = df[
            df["Marks"] > df["SubjectAverage"]
        ].copy()

        return result


    # ============================================================
    # Q12. DEPARTMENT VS GRADE
    # ============================================================

    def department_grade_crosstab(
        self,
        df: pd.DataFrame
    ) -> pd.DataFrame:

        result = pd.crosstab(
            df["Department"],
            df["GradeBand"]
        )

        # Ensure all grade columns are present
        grade_columns = [
            "A+",
            "A",
            "B",
            "C",
            "D",
            "F"
        ]

        result = result.reindex(
            columns=grade_columns,
            fill_value=0
        )

        return result


    # ============================================================
    # Q13. STUDENT RANKING
    # ============================================================

    def student_ranking(
        self,
        df: pd.DataFrame
    ) -> pd.DataFrame:

        summary = (
            df.groupby("StudentID")
            .agg(
                AverageMarks=("Marks", "mean")
            )
            .reset_index()
        )

        # Add names
        names = (
            df[["StudentID", "StudentName"]]
            .drop_duplicates("StudentID")
        )

        summary = summary.merge(
            names,
            on="StudentID",
            how="left"
        )

        # Rank: highest average = 1
        summary["Rank"] = summary["AverageMarks"].rank(
            method="min",
            ascending=False
        ).astype(int)

        # Sort by rank
        summary = summary.sort_values(
            "Rank"
        ).reset_index(drop=True)

        return summary[
            [
                "Rank",
                "StudentID",
                "StudentName",
                "AverageMarks"
            ]
        ]


    # ============================================================
    # Q14. SUBJECT TARGET ANALYSIS
    # ============================================================

    def subject_target_analysis(
        self,
        df: pd.DataFrame
    ) -> pd.DataFrame:

        targets = pd.DataFrame({
            "Subject": [
                "Python",
                "Statistics",
                "Networks",
                "Thermodynamics",
                "Database",
                "Circuits",
                "Design"
            ],
            "TargetMarks": [
                75,
                75,
                70,
                70,
                75,
                75,
                75
            ]
        })

        # Calculate subject average
        subject_avg = (
            df.groupby("Subject")["Marks"]
            .mean()
            .reset_index(name="AverageMarks")
        )

        # Merge average marks with target
        result = subject_avg.merge(
            targets,
            on="Subject",
            how="left"
        )

        # Difference
        result["Difference"] = (
            result["AverageMarks"] -
            result["TargetMarks"]
        )

        # Status
        result["Status"] = np.where(
            result["AverageMarks"] >= result["TargetMarks"],
            "Target Achieved",
            "Below Target"
        )

        return result


    # ============================================================
    # Q15. SUBJECT MARKS VISUALIZATION
    # ============================================================

    def plot_subject_marks(
        self,
        df: pd.DataFrame
    ):

        subject_avg = (
            df.groupby("Subject")["Marks"]
            .mean()
            .sort_values()
        )

        plt.figure(figsize=(10, 6))

        plt.bar(
            subject_avg.index,
            subject_avg.values
        )

        plt.title("Average Marks for Each Subject")
        plt.xlabel("Subject")
        plt.ylabel("Average Marks")

        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.show()


    # ============================================================
    # Q16. ATTENDANCE VS MARKS
    # ============================================================

    def plot_attendance_marks(
        self,
        df: pd.DataFrame
    ):

        plt.figure(figsize=(8, 6))

        plt.scatter(
            df["Attendance"],
            df["Marks"]
        )

        plt.title("Attendance vs Marks")
        plt.xlabel("Attendance")
        plt.ylabel("Marks")

        plt.tight_layout()

        plt.show()

In [2]:
top_5 = (
    student_summary
    .head(5)
)

print("\n========== CHALLENGE 1 ==========")
print(top_5)

NameError: name 'student_summary' is not defined

In [4]:
top_5 = (
    student_summary
    .head(5)
)

print("\n========== CHALLENGE 1 ==========")
print(top_5)

NameError: name 'student_summary' is not defined

In [6]:
students_below_median = (
    df.groupby(
        ["StudentID", "StudentName"]
    )["Attendance"]
    .mean()
    .reset_index(name="AverageAttendance")
)

students_below_median = students_below_median[
    students_below_median["AverageAttendance"]
    < overall_median_attendance
]

print(students_below_median)

NameError: name 'df' is not defined

In [8]:
students_below_median = (
    df.groupby(
        ["StudentID", "StudentName"]
    )["Attendance"]
    .mean()
    .reset_index(name="AverageAttendance")
)

students_below_median = students_below_median[
    students_below_median["AverageAttendance"]
    < overall_median_attendance
]

print(students_below_median)

NameError: name 'df' is not defined

In [10]:
a_plus_percentage = (
    df.groupby("Department")
    .apply(
        lambda x: (x["GradeBand"] == "A+").mean() * 100,
        include_groups=False
    )
    .reset_index(name="APlusPercentage")
)

a_plus_percentage = a_plus_percentage.sort_values(
    "APlusPercentage",
    ascending=False
)

print("\n========== CHALLENGE 4 ==========")
print(a_plus_percentage)

highest_department = a_plus_percentage.iloc[0]

print(
    "\nDepartment with highest A+ percentage:",
    highest_department["Department"]
)

NameError: name 'df' is not defined

In [12]:
above_90 = df[
    df["Attendance"] > 90
]["Marks"].mean()

below_90 = df[
    df["Attendance"] < 90
]["Marks"].mean()

print("\n========== CHALLENGE 5 ==========")

print("Average marks for attendance above 90%:",
      above_90)

print("Average marks for attendance below 90%:",
      below_90)

NameError: name 'df' is not defined